# LangChain + Clembench experiment constructor

**Single configurable notebook** for running LangChain agents on clembench games.

**Only Cell 2 (Config) needs to be changed between experiments.**

| Variable | Description |
|---|---|
| `GAME` | any clembench game (e.g., `taboo`, `referencegame`, etc.|
| `MODEL` | any model (e.g.,`qwen`, `gpt-4o-mini`) |
| `AGENT_TYPE` | a LangChain agent (choose from below or create a custom one) |
| `RUN_ID` | any string — used as the results folder name |
| `NUM_EPISODES` | integer |
| `SINGLE_PASS` | `True` = one pass through instances, `False` = cycle infinitely |

**Credits to**: the clembench team and especially, Philipp Sadler

In [1]:
# ----- CONFIG ---------------------------------------------
GAME         = "wordle"   
MODEL        = "clp-chat"        
AGENT_TYPE   = "LongTermPlanningAgent"  
RUN_ID       = "longterm_langchain_clp_chat"
NUM_EPISODES = 26
SINGLE_PASS  = True
# ----------------------------------------------------------

## 1. Preparation

In [2]:
import json
import httpx
import os

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from playpen.agents import ClemAgent, ClemObservation
from clemcore.clemgame import env, episode_results_folder_callbacks

from clemcore.backends import ModelRegistry
from clemcore.backends import KeyRegistry


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'



In [3]:
CLEMBENCH_HOME = r"C:\Users\white\Desktop\agents_experiments\clembench"
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [4]:
# uncomment and run once to install dependencies, then comment out again
# %pip install -r $CLEMBENCH_HOME/requirements.txt
# %pip install --upgrade ipywidgets jupyter_client clemcore

# Make tqdm usable in Jupyter notebooks
#%pip install --upgrade ipywidgets jupyter_client

In [5]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s {GAME}

clem 3.5.0
Listing all available games (use -v option to see the whole specs)
Found '1' game specs that match the game_selector='{'game_name': 'wordle'}'
wordle:
 	Wordle 5-letter word guessing game.


In [6]:
#register the model if necessary

#registry = ModelRegistry.register("", backend="openai_compatible",
                  #                                                  model_id="")
#registry.get_first_model_spec_that_unify_with("")

In [7]:
#register the API key and url if necessary

#API_KEY = ""
#ORGANIZATION = ""
#BASE_URL ="" 

#KeyRegistry.register("openai_compatible", api_key=API_KEY, organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

In [8]:
def create_model(model_name: str,registry_path: str = "model_registry.json",key_path: str = "key.json", temperature: float = 0.0 ,max_tokens: int = 300) -> ChatOpenAI:
    """Returns a configured ChatOpenAI instance by looking up model_name in model_registry.json.
    Raises ValueError if the model or its required backend credentials are not found.
    SSL verification is disabled for the clp-chat model via the verify_ssl field in model_registry.json
    (add "verify_ssl": False to the model_registry.json to disable other models' verification)
    """
    with open(registry_path) as f:
        registry = json.load(f)

    entry = next((e for e in registry if e["model_name"] == model_name), None)
    if entry is None:
        raise ValueError(f"Model {model_name!r} not found in {registry_path}.")

    backend = entry["backend"]

    with open(key_path) as f:
        keys = json.load(f)

    if backend not in keys:
        raise ValueError(
            f"Backend {backend!r} (required by model {model_name!r}) "
            f"not found in {key_path}."
        )

    credentials = keys[backend]

    verify_ssl = entry.get("verify_ssl", True)
    http_client = httpx.Client(verify=verify_ssl)

    return ChatOpenAI(
        model=entry["model_id"],
        base_url=credentials["base_url"],
        api_key=credentials["api_key"],
        temperature=temperature,
        max_tokens=max_tokens,
        http_client=http_client,
    )

## 2. Agent constructor

In [9]:
# Subagent prompt used by MyAgenticPlayer 
_SUBAGENT_PROMPT = """
You are given a small piece of text which contains gameplay rules. You need to extract the necessary tags
(often written in CAPITAL LETTERS), so that the player can use them for the answer.
Do not output any text apart from the tag(s). Example IO pair:

INPUT:
Let's play a guessing game! Your task is to answer the other player's questions. Based on your knowledge
of the word: $TARGET WORD$, respond to the following questions or guesses. Limit your response to only
'yes' or 'no' with no explanation or other words. Never reveal the answer in your response.

You must reply using the format below and DO NOT ADD ANY TEXT OTHER THAN THIS:

ANSWER: <some text>

Target Word: $TARGET WORD$

OUTPUT:
ANSWER:

If you identified no tags, please return NO TAG as an answer.
"""

In [10]:
class CoreToolsAgent(ClemAgent):
    """Agent with remember / recall / observe / get_observations tools."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self.store = {}
        self._tool_observations = []

        tools = [
            self._remember_rules(),
            self._recall_rules(),
            self._observe_game(),
            self._get_game_observations(),
        ]

        system_prompt = """You are a professional game-playing agent. You have memory tools to help you play any game effectively.

  STRATEGY: follow this order every turn:

  TURN 1 (rules turn):
    1. Store the goal, required response format, and constraints using remember_rules().
    2. Give your first response in the required format.

  TURN 2+ (every subsequent turn):
    1. Call observe_game() to record the new information you just received.
    2. Call get_game_observations() to review what has already happened.
    3. Call recall_rules() to keep to the game rules.
    4. Give your response based on the full picture.

  Never skip steps 1-2 on turn 2+. Observations are required before acting."""

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def reset(self):
        super().reset()
        self.episode += 1
        self.store.clear()
        self._tool_observations.clear()

    def get_memory_snapshot(self) -> list:
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def _remember_rules(self):
        store = self.store
        @tool
        def remember_rules(key: str, value: str) -> str:
            """
            Store any important information concerning the game rules.

            Examples:
                remember_rules("goal", "describe the target without forbidden words")
                remember_rules("format", "CLUE: <text>")
                remember_rules("target", "first grid")
                remember_rules("forbidden", "cat, dog, pet")
            """
            store[key] = value
            return f"Stored: {key} = {value}"
        return remember_rules

    def _recall_rules(self):
        store = self.store
        @tool
        def recall_rules(key: str = "") -> str:
            """
            Retrieve stored information about the game rules.

            Args:
                key: Specific key, or empty for everything
            """
            if not store:
                return "Memory empty."
            if key and key in store:
                return f"{key}: {store[key]}"
            return "\n".join(f"- {k}: {v}" for k, v in store.items())
        return recall_rules

    def _observe_game(self):
        observations = self._tool_observations
        @tool
        def observe_game(observation: str) -> str:
            """
            Note something important you noticed during the game.

            Examples:
                observe("Grid 1 has a red circle")
                observe("The clue mentions 'round shape'")
                observe("Player said 'no' to animal question")
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return observe_game

    def _get_game_observations(self):
        observations = self._tool_observations
        @tool
        def get_game_observations() -> str:
            """Get all observations you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations))
        return get_game_observations

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

In [11]:
class TagExtractorAgent(ClemAgent):
    """Agent that uses an extract_tags subagent to parse game rules on first turn."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.model = model
        self.memory = InMemorySaver()
        self.base_thread_id = thread_id
        self.episode = 0

        @tool
        def extract_tags(initial_prompt: str) -> str:
            """Extract the response tags that are necessary for the player from the game rules."""
            subagent = create_agent(model=self.model, tools=[], system_prompt=_SUBAGENT_PROMPT)
            result = subagent.invoke({"messages": [{"role": "user", "content": initial_prompt}]})
            final = next(m for m in reversed(result["messages"]) if isinstance(m, AIMessage))
            return final.content

        self.agent = create_agent(
            model=self.model,
            tools=[extract_tags],
            checkpointer=self.memory,
            system_prompt=(
                "You're a professional agent game player. You're going to play a game. There is a helpful tool called extract_tags that identifies the required response format. On your first turn, call extract_tags and use the result to format all future responses."
            ),
        )

    def reset(self):
        super().reset()
        self.episode += 1

    def get_memory_snapshot(self) -> list:
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def act(self, last: ClemObservation) -> str:
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}},
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

In [12]:
class TwoToolsAgent(ClemAgent):
    """Agent with perception tools and a short-term memory component.
       As LLMs have no sensors, they might need some additional grounding. 
       While clembench provides an available action space, whereas the tools will provide an observation storage.
       It is expected that the agents will build upon this perception and take less "wrong" action,
       as they will have less outdated/hallucinated 'knowledge'.
    """

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self.store = {}
        self._tool_observations = []

        tools = [
            self._observe_game(),
            self._get_game_observations(),
        ]

        system_prompt = """You are a professional game-playing agent. You have two memory tools to help you play any game effectively.

  STRATEGY: follow this order every turn:

  TURN 1 (You recieve the game rules and the general outline):
    Do not use any tools.

  TURN 2+:
    1. Call observe_game() to record something important you noticed during the game.
    2. Call get_game_observations() to review what has already happened.
    You MUST call observe_game() and get_game_observations() before you give the answer! Never skip steps 1-2 on turn 2+."""

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def reset(self):
        super().reset()
        self.episode += 1
        self.store.clear()
        self._tool_observations.clear()

    def get_memory_snapshot(self) -> list:
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        return state.get("channel_values", {}).get("messages", [])

    def _observe_game(self):
        observations = self._tool_observations
        @tool
        def observe_game(observation: str) -> str:
            """
            Note something important you noticed during the game.

            Examples:
                observe("Grid 1 has a red circle")
                observe("The clue 'round shape' given by me misleaded the other player")
                observe("Player said 'no' to animal question")
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return observe_game

    def _get_game_observations(self):
        observations = self._tool_observations
        @tool
        def get_game_observations() -> str:
            """Get all observations you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations))
        return get_game_observations

    def act(self, last: ClemObservation) -> str:
        print("DEBUG:", self.history)
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
        return "(no response)"

In [13]:
class LongTermPlanningAgent(ClemAgent):
    """Agent with long term memory for planning its actions."""

    def __init__(self, model: ChatOpenAI, thread_id: str = "default"):
        super().__init__()
        self.base_thread_id = thread_id
        self.episode = 0
        self.memory = InMemorySaver()
        self.model = model
        self._tool_observations = []

        tools = [
            self._write_trajectory(),
            self._get_written_trajectories(),
        ]

        system_prompt = """You're a professional game player with planning tools.

    STRATEGY:

    On the FIRST turn: 
    1. Call write_trajectory() to save what you are about to answer.
    
    At the START of every subsequent episode:
    1. Call get_written_trajectories() to recall lessons from past episodes.
    2. Apply that knowledge to play better.

    At the END of every subsequent episode (just before you make your final move):
    1. Call write_trajectory() to save what you learned from the past interaction (What have you attempted? Was it successful or not?).
    Make your move in the format defined by the game rules. Do NOT output anything else!
    """

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            checkpointer=self.memory,
            system_prompt=system_prompt,
        )

    def _write_trajectory(self):
        observations = self._tool_observations
        @tool
        def write_trajectory(observation: str) -> str:
            """
            Promptly reason about your performance throughout the past rounds and make notes for yourself.
            Example: "tried Option 1, unsuccessful; game proceeds; need to observe the remaining options
            """
            observations.append(observation)
            return f"Noted: {observation}"
        return write_trajectory

    def _get_written_trajectories(self):
        observations = self._tool_observations
        @tool
        def get_written_trajectories() -> str:
            """Get all strategies and trajectories you've noted during the game."""
            if not observations:
                return "No observations yet."
            return "\n".join(f"{i+1}. {o}" for i, o in enumerate(observations[-10:]))
        return get_written_trajectories

    def reset(self):
        super().reset()
        self.episode += 1

    def get_memory_snapshot(self) -> list:
        """Return the current LangGraph message history for this episode (last 10 messages)."""
        config = {"configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"}}
        state = self.memory.get(config)
        if state is None:
            return []
        msgs = state.get("channel_values", {}).get("messages", [])
        return msgs[-10:]

    def get_longterm_snapshot(self) -> dict:
        """Return the long-term memory state (all observations, not wiped between episodes)."""
        return {
            "observations": list(self._tool_observations),
        }

    def act(self, last: ClemObservation) -> str:
        memory_summary = ""
        if self._tool_observations:
            memory_summary = "STRATEGIES AND OBSERVATIONS FROM PAST EPISODES:\n"
            memory_summary += "Past observations:\n" + "\n".join(f"- {o}" for o in self._tool_observations[-10:])
        result = self.agent.invoke(
            {"messages": [{"role": "user", "content": last.content + "\n\n" + memory_summary}]},
            config={
                "configurable": {"thread_id": f"{self.base_thread_id}_ep{self.episode}"},
                "recursion_limit": 100,
            },
        )
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and msg.content:
                return msg.content
                
        return "(no response)"

## 3. Experiment setup

In [14]:
def make_agent(agent_type: str, model: ChatOpenAI, thread_id: str) -> ClemAgent:
    """Instantiate an agent by name."""
    if agent_type == "CoreToolsAgent":
        return CoreToolsAgent(model=model, thread_id=thread_id)
    elif agent_type == "TagExtractorAgent":
        return TagExtractorAgent(model=model, thread_id=thread_id)
    elif agent_type == "TwoToolsAgent":
        return TwoToolsAgent(model=model, thread_id=thread_id)
    elif agent_type == "LongTermPlanningAgent":
        return LongTermPlanningAgent(model=model, thread_id=thread_id)
    else:
        raise ValueError(f"Unknown agent type: {agent_type!r}.")

In [15]:
callbacks = episode_results_folder_callbacks(
    run_dir=RUN_ID,
    result_dir_path="playpen-records",
    player_model_infos=f"{AGENT_TYPE}-{MODEL}",
)

game_env = env(GAME, single_pass=SINGLE_PASS, callbacks=callbacks)
#removed reset

print("roles:", game_env.unwrapped.game_benchmark.game_spec["roles"])

2026-04-10 14:24:16,501 - clemcore.cli - INFO - Found '1' game matching the game_selector="wordle"
2026-04-10 14:24:16,504 - clemcore.cli - INFO - {
  "game_name": "wordle",
  "description": "Wordle 5-letter word guessing game.",
  "main_game": "wordle",
  "players": 1,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "2.0",
    "3.0"
  ],
  "regression": "small",
  "roles": [
    "Guesser"
  ],
  "game_path": "C:\\Users\\white\\Desktop\\agents_experiments\\clembench\\wordle"
}
2026-04-10 14:24:16,506 - clemcore.run - INFO - Loading game benchmark for wordle
2026-04-10 14:24:16,531 - clemcore.run - INFO - Loading game benchmark for wordle took: 0:00:00.023084
2026-04-10 14:24:16,559 - clemcore.run - INFO - Prepared instance queue for wordle using 2 experiments ['high_frequency_words_no_clue_no_critic', 'medium_frequency_words_no_clue_no_critic'] and 20 instances in total.
2026-04-10 14:24:16,561 - clemcore.run - INFO - Detected single_pass=True, stopping after fir

In [16]:
model = create_model(MODEL)

roles = game_env.unwrapped.game_benchmark.game_spec["roles"]

if len(roles) == 1 or GAME == "privateshared":
    player_0 = make_agent(AGENT_TYPE, model, thread_id="player0")
    learner_agents = [player_0]
    agent_mapping = {"player_0": player_0}
else:
    player_0 = make_agent(AGENT_TYPE, model, thread_id="player0")
    player_1 = make_agent(AGENT_TYPE, model, thread_id="player1")
    learner_agents = [player_0, player_1]
    agent_mapping = {"player_0": player_0, "player_1": player_1}

print("Agent mapping:", {k: type(v).__name__ for k, v in agent_mapping.items()})

Agent mapping: {'player_0': 'LongTermPlanningAgent'}


## 4. Run game

In [17]:
import json as _json
from pathlib import Path
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def serialize_messages(msgs: list) -> list:
    """Convert LangChain messages to plain dicts for JSON serialization."""
    out = []
    for m in msgs:
        out.append({
            "type": type(m).__name__,
            "content": m.content,
            "tool_calls": getattr(m, "tool_calls", []),
        })
    return out

memory_log_dir = Path("playpen-records") / RUN_ID / GAME / "memory_logs"   #NB: kept separately from the results!
memory_log_dir.mkdir(parents=True, exist_ok=True)

all_episodes_data = []

for episode in range(NUM_EPISODES):
    game_env.reset()
    for agent in learner_agents:
        agent.reset()

    episode_memory_log = []

    context_response_pairs = []
    for step_idx, agent_id in enumerate(game_env.agent_iter()):
        context, reward, termination, truncation, info = game_env.last()
        if termination or truncation:
            response = None
        elif agent_id in agent_mapping:
            response = agent_mapping[agent_id](context)
        else:
            response = game_env.unwrapped.player_by_agent_id[agent_id](context)
        #print(f"  [obs:{agent_id}] {context['content'][:100]!r}")
        context_response_pairs.append((agent_id, context, response, reward))
        game_env.step(response)

        

        # snapshot memory for the agent that just acted
        agent = agent_mapping.get(agent_id)
        if agent is not None and hasattr(agent, "get_memory_snapshot"):
            msgs = agent.get_memory_snapshot()
            snapshot = {
                "step": step_idx,
                "agent_id": agent_id,
                "thread_id": agent.base_thread_id,
                "messages": serialize_messages(msgs),
            }
            episode_memory_log.append(snapshot)
            print(f"  [memory:{agent.base_thread_id}] {len(msgs)} messages: "
                  + " | ".join(f"{type(m).__name__}({m.content[:40]!r})" for m in msgs))

    # write memory log for this episode
    log_path = memory_log_dir / f"episode_{episode + 1:04d}.json"
    with open(log_path, "w") as f:
        _json.dump(episode_memory_log, f, indent=2, default=str)

    # write long-term memory
    for agent in learner_agents:
        if hasattr(agent, "get_longterm_snapshot"):
            lt_path = memory_log_dir / f"longterm_{agent.base_thread_id}.json"
            with open(lt_path, "w") as f:
                _json.dump(agent.get_longterm_snapshot(), f, indent=2, default=str)

    all_episodes_data.append(context_response_pairs)
    print(f"Episode {episode + 1}/{NUM_EPISODES} completed — {len(context_response_pairs)} steps")

print(f"\nAll episodes done. Memory logs written to: {memory_log_dir}")

  [memory:player0] 2 messages: HumanMessage('You are a language wizard who likes to g') | AIMessage('explanation: I will start with a common ')
  [memory:player0] 7 messages: HumanMessage('You are a language wizard who likes to g') | AIMessage('explanation: I will start with a common ') | HumanMessage('guess_feedback: a<yellow> d<yellow> i<re') | AIMessage('') | ToolMessage('No observations yet.') | ToolMessage("Noted: First guess: 'adieu' resulted in ") | AIMessage("explanation: Based on the feedback, 'a',")
  [memory:player0] 10 messages: AIMessage('explanation: I will start with a common ') | HumanMessage('guess_feedback: a<yellow> d<yellow> i<re') | AIMessage('') | ToolMessage('No observations yet.') | ToolMessage("Noted: First guess: 'adieu' resulted in ") | AIMessage("explanation: Based on the feedback, 'a',") | HumanMessage('guess_feedback: d<red> e<yellow> a<green') | AIMessage('') | ToolMessage("Noted: Second guess: 'deads' resulted in") | AIMessage("explanation: Based on the 

StopIteration: 

In [ ]:
# Display the last episode's steps
last_episode = all_episodes_data[-1]
print(f"Last episode: {len(last_episode)} steps")
print("-" * 60)
for idx, (agent_id, context, response, reward) in enumerate(last_episode):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"  Agent({agent_id}) <- Context: {context}")
    print(f"  Agent({agent_id}) -> Response: {response}")
    print("-" * 60)

## 5. After the game

In [ ]:
results_dir = callbacks.callbacks[0].results_folder.results_dir_path
run_dir     = callbacks.callbacks[0].results_folder.run_dir
print(f"Results saved to: {results_dir}")
print(f"Run dir:          {run_dir}")
print()
print("To score results:")
print(f"  clem score -g {GAME} -r playpen-records") #or -r PATH_TO_FOLDER
print(f"  clem eval -r playpen-records")   #or -r PATH_TO_FOLDER

## 6. For debugging (this will enable the content of every LLM call)

In [ ]:
"""
from langchain_core.callbacks import BaseCallbackHandler

class MessageSpy(BaseCallbackHandler):
      def on_chat_model_start(self, serialized, messages, **kwargs):
          print("\n=== MESSAGES TO LLM ===")
          for msg in messages[0]:
              print(f"  {type(msg).__name__}: {msg.content}")
          print("======================\n")
"""

# AND add this line to the act method:
#                bla bla recursion limit = 100,
#                "callbacks": [MessageSpy()],